# Lab 07-03 — LLM-as-judge vs reference: is the judge trustworthy?

**Track 07 · Evaluation** — LLM-as-judge is cheap and scales, but it is a MODEL making a judgment — it can be sycophantic, pattern-match to phrasing, or disagree with the ground truth in systematic ways. This lab applies the repo's standing rule (AGENTS.md #6): before an LLM judge is trusted, cross-check it against a reference-based metric and measure agreement with Cohen's kappa.

This notebook is **self-contained**: it imports LangChain, sentence-transformers, faiss, and Groq directly — no repo component library. Every block of the pipeline is built right here: the BGE embedder, the FAISS index, the top-3 retriever, the answer generator, the LLM judge, the reference rater, and the kappa statistic — all appear as plain code in the cells below, which is exactly how the shared components in `src/` work underneath.

Setup: 15 questions from rag-mini-wikipedia. Each answer is judged "correct?" by two independent raters:

* **LLM judge** — ChatGroq (the same hosted Llama model the generator uses), shown the question, the answer and the gold reference, asked for a binary correct/incorrect verdict (temperature 0). This is the judge whose trustworthiness we are auditing.
* **Reference-based rater** — the embedding cosine between the answer and the gold reference, binarized at a threshold. Deterministic, no model opinion, only geometric similarity to the human gold.

Cohen's kappa (hand-rolled inline — the formula is the point) then answers: beyond the agreement you would expect by chance, how much do the two raters agree? The Landis-Koch scale turns the number into words (0.61+ substantial). A low kappa means the LLM judge is NOT a drop-in replacement for the reference metric — exactly the failure this check exists to catch.

The pipeline, drawn inline:

```text
rag-mini passages (3,200)
  -> BGE embed (HuggingFaceEmbeddings) + FAISS index
  -> retrieve top-3 per question (15 questions)
  -> ChatGroq generates the answer
  -> rate twice: LLM judge verdict + reference cosine verdict
  -> observed agreement + Cohen's kappa (hand-rolled)
  -> verification gate (--verify)
```


## Setup

One prerequisite must hold before this notebook will run:

- **rag-mini-wikipedia on disk** — `Data/corpus/rag-mini-wikipedia/` (`passages.parquet` + `test.parquet`), already fetched by the repo's manifest-verified fetchers.
- **`GROQ_API_KEY` in the repo-root `.env`** — the generator and the judge both run on Groq's hosted Llama model; the imports cell loads the key via python-dotenv.

No repo imports are needed: everything this notebook uses comes from `langchain-core`, `langchain-huggingface`, `langchain-community`, `langchain-groq`, `sentence-transformers`, `faiss-cpu`, `pandas`, and `python-dotenv`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`):
#   sentence-transformers -> local BGE embeddings (HuggingFaceEmbeddings)
#   langchain-huggingface -> the HuggingFaceEmbeddings wrapper
#   langchain-community   -> the FAISS vector store
#   faiss-cpu             -> the FAISS index
#   langchain-groq        -> ChatGroq (generator + judge)
#   python-dotenv         -> loads GROQ_API_KEY from the repo-root .env
#   pandas                -> reads the passages/test.parquet corpus
%pip install -q sentence-transformers langchain-huggingface langchain-community faiss-cpu langchain-groq python-dotenv pandas


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path

import pandas as pd  # noqa: E402  (reads the parquet corpus)

# LangChain + sentence-transformers + faiss + Groq — the only libraries this
# notebook needs. Nothing is imported from the repo's src/ component library.
from dotenv import load_dotenv  # noqa: E402
from langchain_community.vectorstores import FAISS  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from langchain_core.embeddings import Embeddings  # noqa: E402
from langchain_groq import ChatGroq  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)
load_dotenv(REPO_ROOT / ".env")  # GROQ_API_KEY lives in the repo-root .env


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `RAG_MINI` points at the rag-mini-wikipedia subset already on disk; `N_SAMPLE = 15` bounds the judged questions (15 Groq generations + 15 judge calls — the LLM leg dominates the runtime); `TOP_K = 3` is the context fed to the generator; `BGE_MODEL_NAME` selects the local embedder; `GROQ_MODEL_NAME` selects the hosted generator/judge (the same model the lab's `GroqLLM` defaults to); `COSINE_THRESHOLD = 0.5` is the reference rater's decision boundary — cosine >= 0.5 means "correct".


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the experiment
# --------------------------------------------------------------------------
RAG_MINI = Path("Data/corpus/rag-mini-wikipedia")
PASSAGES_PATH = RAG_MINI / "passages.parquet"
TEST_PATH = RAG_MINI / "test.parquet"
N_SAMPLE = 15  # judged questions (hosted LLM calls are the slow leg)
TOP_K = 3  # context chunks fed to the generator
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"  # local embedder (cached)
GROQ_MODEL_NAME = "llama-3.3-70b-versatile"  # generator + judge (lab default)
COSINE_THRESHOLD = 0.5  # reference rater: cosine >= this => "correct"


## 2. Load — passages + test QA

Two parquet files from rag-mini-wikipedia. The passages file has a **single `passage` column** — the loader reads it as plain text. The gold answers in `test.parquet` are terse — often just "yes", "no", or a number — which is exactly why the reference rater below uses embedding cosine instead of brittle exact-match.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — passages + test QA
# --------------------------------------------------------------------------
def load_passages(path: Path) -> tuple[list[str], list[str]]:
    """Return (doc_texts, doc_ids) for every passage."""
    df = pd.read_parquet(path)
    texts = [str(row["passage"]).strip() for _, row in df.iterrows()]
    ids = [str(i) for i in range(len(df))]
    return texts, ids


def load_test_qa(path: Path) -> list[dict]:
    """Return [{"question": ..., "answer": ...}] from test.parquet."""
    df = pd.read_parquet(path)
    return [{"question": r["question"], "answer": r["answer"]}
            for _, r in df.iterrows()]


def cosine_similarity(a: list[float], b: list[float]) -> float:
    """Cosine similarity between two vectors (0.0 if either is zero)."""
    dot = sum(x * y for x, y in zip(a, b))
    na = sum(x * x for x in a) ** 0.5
    nb = sum(x * x for x in b) ** 0.5
    if na == 0.0 or nb == 0.0:
        return 0.0
    return dot / (na * nb)


## 3. The two raters — judge verdict vs reference verdict

Two independent answers to "is this answer correct?" for the same 15 questions. The **LLM judge** reads the question, the answer and the gold reference and returns a binary verdict (JSON `{"correct": true|false}`); the **reference rater** is pure geometry — the cosine between the answer embedding and the gold-reference embedding, binarized at `COSINE_THRESHOLD`. One is a model with opinions; the other is a deterministic formula. The whole point of the lab is to measure how much they agree.


In [ ]:
# --------------------------------------------------------------------------
# 3. The two raters — judge verdict vs reference verdict
# --------------------------------------------------------------------------
def judge_verdict(judge, question: str, answer: str, reference: str) -> int:
    """LLM judge: is the answer correct? -> 1 correct, 0 incorrect.

    JSON schema ``{"correct": true|false}``; accepts bool or "yes"/"true"
    strings from the model.
    """
    instruction = (
        "You are an answer-quality judge. Decide whether the answer is "
        "CORRECT given the gold reference. Output ONLY JSON."
    )
    prompt = (
        f"Question: {question}\n\nAnswer: {answer}\n\n"
        f"Gold reference: {reference}\n\n"
        'Return JSON: {"correct": true} or {"correct": false}'
    )
    result = judge.judge(instruction, prompt)
    raw = result.get("correct", False)
    if isinstance(raw, bool):
        return 1 if raw else 0
    return 1 if str(raw).strip().lower() in ("yes", "true", "1", "correct") else 0


def reference_verdict(cosine: float) -> int:
    """Reference-based rater: cosine against gold >= threshold -> 1 else 0."""
    return 1 if cosine >= COSINE_THRESHOLD else 0


## 4. The judge + Cohen's kappa — hand-rolled inline

The lab imports `LLMJudge` and `EvaluationHarness.kappa` from the shared evaluation block. Here both are built by hand with the **same rubric and scale**: `InlineJudge` wraps `ChatGroq` — `judge(instruction, prompt)` asks the model for a JSON object, strips a markdown code fence if the model wraps its answer, retries once on parse failure, and returns `{"error": ...}` when both attempts fail; `embed(texts)` returns local BGE vectors (the same model family the corpus was indexed with). `cohens_kappa` is the from-scratch statistic: observed agreement minus the agreement expected by chance, normalized by the room above chance. The formula — and the chance-correction it encodes — is the point of the cell.


In [ ]:
# --------------------------------------------------------------------------
# 4. The judge + Cohen's kappa — hand-rolled inline (same contract)
# --------------------------------------------------------------------------
def _strip_code_fence(text: str) -> str:
    """Remove a surrounding markdown code fence (```json ... ```)."""
    lines = text.strip().splitlines()
    if lines and lines[0].startswith("```"):
        lines = lines[1:]
    if lines and lines[-1].strip() == "```":
        lines = lines[:-1]
    return "\n".join(lines).strip()


class InlineJudge:
    """LLM-as-judge over ChatGroq + local BGE embeddings (hand-rolled).

    Mirrors the shared LLMJudge contract the lab uses: ``judge()`` returns
    the parsed JSON dict (one retry on parse failure, then {"error": ...});
    ``embed()`` returns local BGE vectors.
    """

    def __init__(self, llm: ChatGroq, embedder):
        self.llm = llm
        self.embedder = embedder

    def judge(self, instruction: str, prompt: str) -> dict:
        last_error = ""
        for attempt in range(2):
            full = f"{instruction}\n\n{prompt}\n\nRespond with ONLY a valid JSON object."
            if attempt == 1:
                full += " Respond with ONLY valid JSON."
            try:
                text = self.llm.invoke(full).content
                return json.loads(_strip_code_fence(text))
            except (json.JSONDecodeError, ValueError) as exc:
                last_error = str(exc)
        return {"error": f"inline judge: could not parse JSON after 2 attempts: {last_error}"}

    def embed(self, texts: list[str]) -> list[list[float]]:
        return self.embedder.embed_documents(texts)


def cohens_kappa(labels_a: list[int], labels_b: list[int]) -> float:
    """Cohen's kappa for two binary label lists, computed from scratch.

    kappa = (observed agreement - expected agreement) / (1 - expected).
    Expected agreement is the product of the marginal rates — the
    agreement two raters would reach by chance given how often each
    says "correct".
    """
    if len(labels_a) != len(labels_b):
        raise ValueError("cohens_kappa: label lists must have equal length")
    n = len(labels_a)
    if n == 0:
        return 0.0
    a11 = sum(1 for x, y in zip(labels_a, labels_b) if x == 1 and y == 1)
    a10 = sum(1 for x, y in zip(labels_a, labels_b) if x == 1 and y == 0)
    a01 = sum(1 for x, y in zip(labels_a, labels_b) if x == 0 and y == 1)
    a00 = sum(1 for x, y in zip(labels_a, labels_b) if x == 0 and y == 0)
    po = (a11 + a00) / n
    pe = ((a11 + a10) * (a11 + a01) + (a01 + a00) * (a10 + a00)) / (n * n)
    if pe == 1.0:
        return 0.0
    return (po - pe) / (1 - pe)


def landis_koch(kappa: float) -> str:
    """Human words for a kappa value (Landis & Koch, 1977)."""
    if kappa < 0.0:
        return "poor (worse than chance)"
    if kappa < 0.21:
        return "slight"
    if kappa < 0.41:
        return "fair"
    if kappa < 0.61:
        return "moderate"
    if kappa < 0.81:
        return "substantial"
    return "almost perfect"


## 5. Experiment — retrieve, generate, rate twice

The full pipeline: embed all 3,200 passages with the local BGE model (normalized — BGE requires it for cosine), index them in FAISS through a precomputed-vector passthrough so the embed step and the index step stay separately timed, retrieve the top-3 context per question, let ChatGroq generate an answer, then rate it twice — the LLM judge verdict and the reference cosine verdict — and finally compute observed agreement and Cohen's kappa over the 15 questions.

`device="cpu"` on the embedder mirrors the lab: bulk-embedding 3,200 passages on CPU keeps the run light on the shared machine's GPU. The thread cap below keeps the BLAS/OpenMP footprint small while several track agents share this box.


In [ ]:
# --------------------------------------------------------------------------
# 5. Experiment — retrieve, generate, rate twice
# --------------------------------------------------------------------------
# Several track agents share this machine — cap BLAS/OpenMP threads so the
# BGE embedding step stays light on CPU and memory.
os.environ["OMP_NUM_THREADS"] = "2"
import torch  # noqa: E402
torch.set_num_threads(2)


class _PrecomputedEmbeddings(Embeddings):
    """Hand the store precomputed vectors (looked up BY TEXT, not by order).

    FAISS calls embed_documents once with the full list; the lookup keeps the
    embed step and the index step separately timed, and would stay correct if
    a store batched the call. embed_query is delegated to the real embedder so
    the store's retriever can embed queries.
    """

    def __init__(self, texts: list[str], embeddings: list[list[float]],
                 query_embedder: Embeddings):
        if len(texts) != len(embeddings):
            raise ValueError("texts and embeddings must be parallel lists")
        self._table: dict[str, list[float]] = dict(zip(texts, embeddings))
        self._query_embedder = query_embedder

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        missing = [t for t in texts if t not in self._table]
        if missing:
            raise ValueError(f"{len(missing)} text(s) have no precomputed vector")
        return [self._table[t] for t in texts]

    def embed_query(self, text: str) -> list[float]:
        return self._query_embedder.embed_query(text)


def run_experiment() -> dict:
    passages, passage_ids = load_passages(PASSAGES_PATH)
    test_qa = load_test_qa(TEST_PATH)

    # --- Embed locally (BGE, CPU) and index in-memory -----------------------
    embedder = HuggingFaceEmbeddings(
        model_name=BGE_MODEL_NAME,
        model_kwargs={"device": "cpu"},  # leave the shared GPU alone
        encode_kwargs={"normalize_embeddings": True},  # BGE needs cosine-normalized vectors
    )
    t0 = time.perf_counter()
    vectors = embedder.embed_documents(passages)
    embed_s = time.perf_counter() - t0

    chunks = [
        Document(page_content=t, metadata={"id": cid})
        for t, cid in zip(passages, passage_ids)
    ]
    store = FAISS.from_documents(
        chunks, embedding=_PrecomputedEmbeddings(passages, vectors, embedder)
    )
    retriever = store.as_retriever(search_kwargs={"k": TOP_K})

    # --- Generator + hand-rolled judge (both ChatGroq, temp 0) --------------
    llm = ChatGroq(model=GROQ_MODEL_NAME, temperature=0.0)
    judge = InlineJudge(llm, embedder)

    judge_labels: list[int] = []
    reference_labels: list[int] = []
    rows: list[dict] = []

    t0 = time.perf_counter()
    for item in test_qa[:N_SAMPLE]:
        question, reference = item["question"], item["answer"]
        context = "\n\n".join(d.page_content
                              for d in retriever.invoke(question))
        answer = llm.invoke(
            f"Context:\n{context}\n\nQuestion: {question}\n\n"
            "Answer in one or two complete sentences, stating the key "
            "fact(s) from the context:"
        ).content.strip()

        cosine = cosine_similarity(judge.embed([answer])[0],
                                   judge.embed([reference])[0])
        jv = judge_verdict(judge, question, answer, reference)
        rv = reference_verdict(cosine)
        judge_labels.append(jv)
        reference_labels.append(rv)
        rows.append({
            "question": question,
            "reference": reference,
            "answer": answer,
            "cosine": cosine,
            "judge": jv,
            "reference": rv,
        })
    llm_s = time.perf_counter() - t0

    kappa = cohens_kappa(judge_labels, reference_labels)
    n = len(judge_labels)
    agreement = sum(1 for a, b in zip(judge_labels, reference_labels)
                    if a == b) / n if n else 0.0
    return {
        "rows": rows,
        "indexed": len(passages),
        "embed_s": embed_s,
        "llm_s": llm_s,
        "kappa": kappa,
        "agreement": agreement,
        "judge_labels": judge_labels,
        "reference_labels": reference_labels,
    }


## 6. Demo — print the artifact

`print_demo(exp)` prints the verdict table (judge vs reference per question, with AGREE/DISAGREE marks), the observed agreement, and the kappa with its Landis-Koch words. Watch the gap between the two numbers: observed agreement alone flatters — two raters who both say "correct" most of the time agree by chance. Kappa subtracts that expected agreement, so it is the honest number.


In [ ]:
# --------------------------------------------------------------------------
# 6. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 07-03 — LLM-as-judge vs reference: Cohen's kappa")
    print(f"rag-mini {exp['indexed']} passages, {len(exp['rows'])} questions")
    print("=" * 66)

    print("\n[1] Verdicts per question (judge vs reference-based):")
    for row in exp["rows"]:
        mark = "AGREE" if row["judge"] == row["reference"] else "DISAGREE"
        print(f"    judge={row['judge']} ref={row['reference']} "
              f"cos={row['cosine']:.2f} [{mark}] {row['question'][:50]}")

    k = exp["kappa"]
    print(f"\n[2] Agreement: observed {exp['agreement']:.3f}, "
          f"Cohen's kappa = {k:.3f} -> {landis_koch(k)}")

    print(f"\n[3] Takeaway")
    print("    Observed agreement alone flatters: two raters who both say")
    print("    'correct' most of the time agree by chance. Kappa subtracts")
    print("    that expected agreement, so it is the honest number. A kappa")
    print("    below ~0.6 means the LLM judge is not interchangeable with")
    print("    the reference metric — trust it only after this cross-check,")
    print("    and prefer it as a supplement, not a replacement. Note the")
    print("    threshold choice (cosine >= 0.5) also moves the reference")
    print("    rater's side of the table: kappa audits the PAIR.")


## 7. Verification gate

`verify_gate(exp)` enforces the lab's hard checks: `N_SAMPLE` questions judged, kappa finite and in [-1, 1], one label per question per rater, the reference rater found some correct answers, and — the non-trivial one — **the LLM judge produced some incorrect verdicts**: a judge that says "correct" every time is a yes-machine, not a rater. This is the same gate the CI-style `--verify` run applies; every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 7. Verification gate — run ``python <lab> --verify`` from the repo root
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []
    n = len(exp["rows"])
    k = exp["kappa"]

    checks.append((f"{N_SAMPLE} questions judged (>= 10)", n >= 10))
    checks.append(("kappa is finite and in [-1, 1]",
                   -1.0 <= k <= 1.0 and k == k))
    checks.append(("both verdict lists have one label per question",
                   len(exp["judge_labels"]) == n
                   and len(exp["reference_labels"]) == n))
    checks.append(("reference rater found some correct answers (sum > 0)",
                   sum(exp["reference_labels"]) > 0))
    checks.append(("LLM judge produced some incorrect verdicts (sum < n) "
                   "- judge is not a yes-machine",
                   sum(exp["judge_labels"]) < n))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

Bulk-embedding 3,200 passages on CPU takes a few minutes (no downloads — the BGE model is cached); the 15 ChatGroq generations + 15 judge verdicts follow. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The verdict table (judge vs reference per question), observed agreement, and Cohen's kappa with its Landis-Koch words. Watch the gap between observed agreement and kappa — chance agreement is the difference.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the rag-mini files are intact and `GROQ_API_KEY` is set.


In [ ]:
verify_gate(exp)
